# Step 12: how many clips does the behaviour check need?

Erwyna asked whether training on 2,000 to 3,000 clips in the cloud would make the check more accurate, and what the best amount is.
This notebook answers that with the data we have, instead of guessing:

1. **Learning curves.** Train the same classifier as the app (CLAP embeddings + logistic regression) on 10%, 20%, ... 100% of the training data, five times each, and score on animals it never heard. If the score is still rising at 100%, more clips help; if it is flat, more of the same clips will not.
2. **Clips or animals?** For cats we know which cat made each meow. Train on 2, 4, 8, 12 or all training cats, and see whether accuracy on new cats depends on the number of clips or the number of animals.
3. **Extrapolate.** Fit a simple curve (score = a - b * n^-c) to say roughly how many labelled clips a target score would need.

Everything runs in Colab (free), not on a phone. Same data as steps 1 and 4: CatMeows (440 meows, 21 cats) and EmotionalCanines (1,400 barks).

In [ ]:
import os, glob, json, time, zipfile, subprocess, urllib.request, shutil
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, librosa, torch
from scipy.optimize import curve_fit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold, StratifiedShuffleSplit
from sklearn.metrics import f1_score
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. Data (same as steps 1 and 4)

In [ ]:
if not os.path.isdir("catmeows"):
    rec = json.load(urllib.request.urlopen("https://zenodo.org/api/records/4008297"))
    zf = [f for f in rec["files"] if f["key"].endswith(".zip")][0]
    urllib.request.urlretrieve(zf["links"]["self"], "catmeows.zip")
    zipfile.ZipFile("catmeows.zip").extractall("catmeows")
cat_files = sorted(glob.glob("catmeows/**/*.wav", recursive=True))
cat_y = np.array(["FBI".index(os.path.basename(f).split("_")[0]) for f in cat_files])
cat_g = np.array([os.path.basename(f).split("_")[1] for f in cat_files])

if not os.path.exists("EmotionalCanines/husky_train_labels.csv"):
    shutil.rmtree("EmotionalCanines", ignore_errors=True)
    subprocess.run("GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 https://github.com/tmdang1101/EmotionalCanines", shell=True, check=True)
rows = []
for breed in ["husky", "shiba"]:
    for split in ["train", "test"]:
        df = pd.read_csv(f"EmotionalCanines/{breed}_{split}_labels.csv"); df["split"] = split
        df["path"] = [f"EmotionalCanines/{breed}/{split}/{a}.wav" for a in df.audio_id]; rows.append(df)
dogs = pd.concat(rows, ignore_index=True)
MEDIA = "https://media.githubusercontent.com/media/tmdang1101/EmotionalCanines/main/"
def fetch(p):
    for _ in range(3):
        if os.path.exists(p) and os.path.getsize(p) > 1000: return
        try: urllib.request.urlretrieve(MEDIA + p[len("EmotionalCanines/"):], p)
        except Exception: time.sleep(2)
with ThreadPoolExecutor(8) as ex: list(ex.map(fetch, dogs.path))
def report_label(a, v):
    return 0 if v == "Positive" else 1 if v == "Neutral" else (3 if a == "High" else 2)
dog_y = {"energy": np.array([["Low","Medium","High"].index(a) for a in dogs.arousal]),
         "mood": np.array([["Negative","Neutral","Positive"].index(v) for v in dogs.valence]),
         "report4": np.array([report_label(a, v) for a, v in zip(dogs.arousal, dogs.valence)])}
tr = (dogs.split == "train").values
def load(paths, sr):  # first 10 s, like the phone
    return [librosa.load(p, sr=sr, mono=True)[0][:10 * sr] for p in paths]
print(len(cat_files), "cat clips,", len(dogs), "dog clips")

## 2. CLAP embeddings (the model the phone uses)

In [ ]:
from transformers import ClapModel, ClapProcessor
m = ClapModel.from_pretrained("laion/clap-htsat-unfused").to(device).eval()
p = ClapProcessor.from_pretrained("laion/clap-htsat-unfused")
def emb(a):
    inp = p.feature_extractor([a], sampling_rate=48000, padding="repeatpad", return_tensors="pt")
    with torch.no_grad():
        e = m.audio_projection(m.audio_model(input_features=inp["input_features"].to(device)).pooler_output)
    return torch.nn.functional.normalize(e, dim=-1)[0].cpu().numpy()
t0 = time.time()
Xc = np.stack([emb(a) for a in load(cat_files, 48000)])
Xd = np.stack([emb(a) for a in load(dogs.path, 48000)])
print("embedded", len(Xc) + len(Xd), "clips in", round(time.time() - t0), "s")

## 3. Learning curves

C is fixed at a value that worked in steps 5 to 7 (0.01 for dogs, 0.1 for cats), so the curve shows the effect of data, not of tuning.
Cats: 7 folds by cat; inside each fold the training clips are subsampled. Dogs: the official test dogs; the training clips are subsampled, keeping the label mix.

In [ ]:
def lr(C): return make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, C=C))
FRACS = [0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
SEEDS = 5
curves = {}

def cat_curve():
    rows = []
    for f in FRACS:
        scores = []
        for s in range(SEEDS):
            rng = np.random.default_rng(s)
            pred = np.zeros_like(cat_y)
            for a, b in GroupKFold(7).split(Xc, cat_y, cat_g):
                a = a if f == 1.0 else rng.choice(a, max(30, int(f * len(a))), replace=False)
                pred[b] = lr(0.1).fit(Xc[a], cat_y[a]).predict(Xc[b])
            scores.append(f1_score(cat_y, pred, average="macro"))
        rows.append({"train_clips": int(f * len(cat_y) * 6 / 7), "f1": round(float(np.mean(scores)), 3), "sd": round(float(np.std(scores)), 3)})
        print("cat", rows[-1], flush=True)
    return rows

def dog_curve(task):
    y = dog_y[task]; tri = np.where(tr)[0]; te = np.where(~tr)[0]
    rows = []
    for f in FRACS:
        scores = []
        for s in range(SEEDS):
            if f == 1.0:
                a = tri
            else:
                a = tri[next(StratifiedShuffleSplit(1, train_size=f, random_state=s).split(Xd[tri], y[tri]))[0]]
            scores.append(f1_score(y[te], lr(0.01).fit(Xd[a], y[a]).predict(Xd[te]), average="macro"))
        rows.append({"train_clips": int(len(a)), "f1": round(float(np.mean(scores)), 3), "sd": round(float(np.std(scores)), 3)})
        print("dog_" + task, rows[-1], flush=True)
    return rows

curves["cat"] = cat_curve()
for t in ["energy", "mood", "report4"]:
    curves["dog_" + t] = dog_curve(t)

## 4. Clips or animals? (cats)

In [ ]:
cats = np.unique(cat_g)
by_cats = []
for k in [2, 4, 8, 12, 16]:
    scores, n_clips = [], []
    for s in range(10):
        rng = np.random.default_rng(s)
        test_cats = rng.choice(cats, 4, replace=False)
        rest = np.setdiff1d(cats, test_cats)
        train_cats = rng.choice(rest, min(k, len(rest)), replace=False)
        a = np.isin(cat_g, train_cats); b = np.isin(cat_g, test_cats)
        if len(np.unique(cat_y[a])) < 3: continue
        scores.append(f1_score(cat_y[b], lr(0.1).fit(Xc[a], cat_y[a]).predict(Xc[b]), average="macro"))
        n_clips.append(int(a.sum()))
    by_cats.append({"train_cats": k, "mean_train_clips": int(np.mean(n_clips)), "f1": round(float(np.mean(scores)), 3), "sd": round(float(np.std(scores)), 3)})
    print(by_cats[-1], flush=True)

## 5. Extrapolate: how many labelled clips for a better score?

In [ ]:
def law(n, a, b, c): return a - b * np.power(n, -c)
extrap = {}
for name, rows in curves.items():
    n = np.array([r["train_clips"] for r in rows], float); f = np.array([r["f1"] for r in rows])
    try:
        (a, b, c), _ = curve_fit(law, n, f, p0=[f.max() + 0.05, 1.0, 0.5], bounds=([0, 0, 0.01], [1, 100, 3]), maxfev=20000)
        now = float(n.max())
        extrap[name] = {"ceiling_a": round(float(a), 3), "now_clips": int(now), "f1_now": round(float(law(now, a, b, c)), 3),
                        "f1_at_2x": round(float(law(2 * now, a, b, c)), 3), "f1_at_3000": round(float(law(3000, a, b, c)), 3),
                        "f1_at_10000": round(float(law(10000, a, b, c)), 3)}
    except Exception as e:
        extrap[name] = {"error": str(e)}
    print(name, extrap[name], flush=True)
RESULTS = {"curves": curves, "cats_by_number_of_cats": by_cats, "extrapolation": extrap}
print("RESULTS_JSON", json.dumps(RESULTS))

## How to read this

- A curve that is still climbing at 100% means more labelled clips of the same kind will help; the extrapolation says roughly how much.
- If cats improve with more *cats* but not with more clips from the same cats, then the next data to collect is many different animals, a few clips each, not thousands of clips.
- Only labelled clips help the feeling. Unlabelled pet sounds (for example from YouTube) can train the sound type (bark, purr), but not the feeling, because nobody says how the animal felt.